In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd



df = pd.read_csv('thyroid_cancer_risk_data.csv')
data_a = df["T3_Level"].values

data_b = df["Nodule_Size"].values

risk_flags = ["Radiation_Exposure", "Iodine_Deficiency", "Smoking", "Obesity"]
df["Risk_Factor_Count"] = (df[risk_flags] == "Yes").sum(axis=1)
data_c = df["Risk_Factor_Count"].values

np.random.seed(42)
N_trials = 30
num_chunks = 5000
p_malignant = (df["Diagnosis"] == "Malignant").mean()

# Simulating 5,000 sets of N=30 chunks
trials = np.random.choice([1, 0], size=(num_chunks, N_trials), p=[p_malignant, 1 - p_malignant])
data_d = np.sum(trials, axis=1)

In [ ]:
def para(data,n):
    #normal
    mu = float(np.mean(data))
    sigma = float(np.std(data, ddof=0))
    kc = np.linspace(mu - 3.5 * sigma, mu + 3.5 * sigma, 400)
    pdf = (1.0 / (sigma * np.sqrt(2.0 * np.pi))) * np.exp(-0.5 * ((kc - mu) / sigma) ** 2)
    #laplace
    mu = float(np.median(data))
    b = float(np.mean(np.abs(data - mu)))
    kc = np.linspace(mu - 4.0 * b, mu + 4.0 * b, 400)
    pdf = (1.0 / (2.0 * b)) * np.exp(-np.abs(kc - mu) / b)
    #gamma all positive
    mean_val = float(np.mean(data))
    var_val = float(np.var(data, ddof=1))
    alpha = (mean_val ** 2) / var_val
    beta_param = mean_val / var_val
    kc = np.linspace(1e-4, np.max(data) + 2.0, 400)
    pdf = (beta_param ** alpha / math.gamma(alpha)) * (kc ** (alpha - 1.0)) * np.exp(-beta_param * kc)
    #beta 
    a, b = float(np.min(data)), float(np.max(data))
    span = b - a
    u = np.clip((data - a) / span, 1e-4, 1.0 - 1e-4)
    m_u = float(np.mean(u))
    v_u = float(np.var(u))
    factor = (m_u * (1.0 - m_u) / v_u) - 1.0
    alpha = m_u * factor
    beta_param = (1.0 - m_u) * factor
    kc = np.linspace(a, b, 400)
    u_grid = np.clip((kc - a) / span, 1e-4, 1.0 - 1e-4)
    pdf = ((u_grid ** (alpha - 1.0) * (1.0 - u_grid) ** (beta_param - 1.0)) / beta_func(alpha, beta_param)) / span
    #poison
    lam = float(np.mean(data))
    max_k = int(np.max(data)) + 3
    kc = np.arange(0, max_k + 1)
    pmf = np.array([(lam ** k * np.exp(-lam)) / math.factorial(k) for k in kc])
    #binomial
    p = float(np.mean(data) / n)
    kc = np.arange(0, n + 1)
    pmf = np.array([math.comb(n, int(k)) * (p ** k) * ((1.0 - p) ** (n - k)) for k in kc])
    #multinomial
    categories = np.unique(data)
    K = len(categories)
    pmf = np.array([float(np.mean(data == cat)) for cat in categories])
    kc = np.arange(K)
    #hist(data,density=True)
    #plot(x_a,pdf_a)
